# DermaLens AI · VMamba Tiny S2L5 · Real-only · SMOKE / FULL

**Architecture:** Genuine *VMamba-Tiny (VSSM, 2D Selective Scan)*, not a CNN renamed as Mamba. Official [MzeroMiko/VMamba](https://github.com/MzeroMiko/VMamba) repository's `vmamba_tiny_s2l5` checkpoint (pretrained ImageNet-1K); four state-space stages, depths `[2,2,5,2]`. This uses the one-file official implementation, plus the official optimized selective-scan CUDA (`oflex`) extension. The checkpoint is loaded **before** replacing the 1000-way classifier with our six-class head.

**Real dataset:** Kaggle `shanmukhadattaboda/facial-dermatology-image-dataset`, frozen `train` / `validation` / `test` and six classes.

**Research question:** Do Visual State-Space Models achieve competitive macro-F1 on six-class facial dermatology images compared with CNNs and self-supervised transformers?

**Modes:** `SMOKE` (default) checks CUDA kernel build/import, pretrained weight load, forward/backward through the genuine VMamba, each fine-tuning stage, checkpoint roundtrip, metrics and ZIP export on a small class-balanced subset. `FULL` trains the whole dataset using the same staged strategy. **SMOKE scores are not research results.**

**CUDA requirement:** VMamba is the most environment-sensitive baseline. Kaggle's CUDA toolchain and PyTorch must support compiling the official selective-scan extension. **A successful mock/local CPU test does not validate this.** The notebook *fails clearly* if the CUDA kernel is absent or incompatible, and never silently substitutes a random-weight model or a fake VMamba. Internet access is required to clone the official repository, install dependencies and download its genuine ImageNet checkpoint.

**Training protocol:** Head-only → final VSSM stage (`layers.3`) → last two stages (`layers.2` + `layers.3`) → optional low-learning-rate entire backbone. Best validation macro-F1 chooses the model; only the selected model is evaluated on the held-out test. Geometric augmentation is training-only; images preserve their aspect ratio. Optional full stage runs during SMOKE for code coverage but stays disabled in FULL by default.

**Portable outputs:** State dictionary, exact architecture-source files, checkpoint/source metadata, preprocessing/class map, per-image predictions, charts, logs and ZIP. Later GPU evaluation notebooks must reinstall/build a matching selective-scan CUDA extension; the checkpoint cannot run without the architecture and a compatible implementation.

## Cell 1 — Validate Kaggle runtime and imports

Keep Kaggle's own CUDA-linked `torch` and `torchvision` intact. Install only lightweight Python support packages if missing. We compile **VMamba's official selective-scan extension** later after checking CUDA tooling. Do not use a generic Mamba architecture or silently fall back to CPU.

In [ ]:
import os, sys, json, csv, time, random, hashlib, logging, traceback, shutil, zipfile, math
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from contextlib import nullcontext
import importlib.util, subprocess

NEEDED = {"timm": "timm==1.0.20", "fvcore": "fvcore", "einops": "einops", "ninja": "ninja", "packaging": "packaging"}
missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageFile, UnidentifiedImageError
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import timm
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_recall_fscore_support
)

ImageFile.LOAD_TRUNCATED_IMAGES = False
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False  # custom CUDA VMamba kernel; determinism not guaranteed
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0])
print("Torch:", str(torch.__version__), "TorchVision:", __import__("torchvision").__version__, "timm:", timm.__version__)
print("Device:", DEVICE, "torch CUDA:", torch.version.cuda)
if DEVICE.type != "cuda":
    raise RuntimeError("VMamba requires a Kaggle GPU T4 runtime: Settings → Accelerator → GPU T4 (not CPU).")
print("GPU:", torch.cuda.get_device_name(0), "memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

## Cell 2 — Central experiment settings

Leave `RUN_MODE = "SMOKE"` for the first Kaggle run. **After it passes, change that single line to `RUN_MODE = "FULL"` and Save & Run All.** The dataset resolver uses the nested Kaggle account path automatically; never attach the old unsplit collection. `full_fine_tune=False` by default avoids unnecessary T4 memory cost.

In [ ]:
from dataclasses import dataclass, asdict

# <<< CHANGE ONLY THIS LINE AFTER THE SMOKE RUN PASSES >>>
RUN_MODE = "SMOKE"  # Options: "SMOKE" or "FULL"
if RUN_MODE not in {"SMOKE", "FULL"}:
    raise ValueError("RUN_MODE must be exactly SMOKE or FULL")

@dataclass(frozen=True)
class Config:
    dataset_slug: str = "datasets/shanmukhadattaboda/facial-dermatology-image-dataset"
    output_name: str = "vmamba_tiny_s2l5_real_progressive_v1"
    model_name: str = "vmamba_tiny_s2l5"
    imagenet_weights: str = "official VMamba-Tiny s2l5 ImageNet-1K epoch 262"
    image_size: int = 224
    batch_size: int = 4
    num_workers: int = 2
    epochs_head: int = 4
    epochs_stage4: int = 8
    epochs_stages3_4: int = 8
    full_fine_tune: bool = False
    require_cuda_kernel: bool = True
    epochs_full: int = 5
    patience: int = 4
    min_epochs_per_stage: int = 2
    use_amp: bool = True
    weight_decay: float = 1e-4
    grad_clip_norm: float = 1.0
    use_class_weights: bool = True
    verify_full_sha256: bool = True
    max_suspicious_images_report: int = 100
    seed: int = SEED
    # SMOKE mode only. Sample counts are capped by availability in each class.
    smoke_train_per_class: int = 2
    smoke_val_per_class: int = 1
    smoke_test_per_class: int = 1
    smoke_batch_size: int = 1
    smoke_epochs_per_stage: int = 1
    smoke_test_optional_full_stage: bool = True

CFG = Config()
SMOKE = (RUN_MODE == "SMOKE")
CLASSES = ["Acne", "Fungal_infection", "Hyper_Pigmentation", "Normal_Skin", "Skin_cancer", "Vitiligo"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASSES)}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
RUN_OUTPUT_NAME = CFG.output_name + ("_SMOKE" if SMOKE else "_FULL")
OUT = OUTPUT_ROOT / RUN_OUTPUT_NAME
for folder in (OUT, OUT / "checkpoints", OUT / "plots", OUT / "reports"):
    folder.mkdir(parents=True, exist_ok=True)
(OUT / "config.json").write_text(json.dumps({**asdict(CFG), "classes": CLASSES, "class_to_idx": CLASS_TO_IDX, "run_mode": RUN_MODE,
                                                "effective_batch_size": CFG.smoke_batch_size if SMOKE else CFG.batch_size}, indent=2), encoding="utf-8")
(OUT / "class_to_idx.json").write_text(json.dumps(CLASS_TO_IDX, indent=2), encoding="utf-8")

LOGGER = logging.getLogger("dermalens_vmamba_tiny_s2l5")
LOGGER.setLevel(logging.INFO)
LOGGER.propagate = False
for old_handler in list(LOGGER.handlers):
    LOGGER.removeHandler(old_handler)
fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(OUT / "training.log", encoding="utf-8")
file_handler.setFormatter(fmt)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(fmt)
LOGGER.addHandler(file_handler)
LOGGER.addHandler(stream_handler)

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, default=str, allow_nan=False), encoding="utf-8")

LOGGER.info("Configuration saved: %s", OUT / "config.json")
LOGGER.info("RUN MODE: %s (SMOKE results are NOT scientifically valid)", RUN_MODE)
LOGGER.info("Full fine-tuning enabled in FULL mode: %s", CFG.full_fine_tune)

## Cell 3 — Locate the finalized Kaggle dataset

Search for exactly the `facial-dermatology-image-dataset` name, including `/kaggle/input/datasets/shanmukhadattaboda/...`. Do not accidentally select the older *facial-skin-conditions-real-image-collection* folder, which lacks frozen splits.

In [ ]:
def has_splits(path: Path) -> bool:
    return all((path / name).is_dir() for name in ("train", "validation", "test"))

def resolve_dataset_root():
    search_base = Path("/kaggle/input")
    preferred = search_base / CFG.dataset_slug
    candidates = [preferred]
    candidates.append(search_base / "facial-dermatology-image-dataset")
    # Some Kaggle account configurations store the dataset in a nested dataset path.
    if search_base.is_dir():
        candidates.extend(search_base.glob("**/facial-dermatology-image-dataset"))
    # Test direct split root first; allow one or two extra folders within a mounted dataset.
    found = []
    for folder in candidates:
        if folder.is_dir():
            if has_splits(folder):
                found.append(folder.resolve())
            else:
                for inner in folder.rglob("train"):
                    if inner.is_dir() and has_splits(inner.parent):
                        found.append(inner.parent.resolve())
    unique = sorted(set(found), key=str)
    if len(unique) != 1:
        visible = [str(p) for p in search_base.iterdir()] if search_base.is_dir() else []
        raise FileNotFoundError(
            "Expected exactly one attached dataset with train/validation/test. "
            f"Found: {[str(p) for p in unique]}. Visible /kaggle/input: {visible}. "
            "Add the Kaggle dataset shanmukhadattaboda/facial-dermatology-image-dataset."
        )
    return unique[0]

DATA_ROOT = resolve_dataset_root()
SPLIT_DIRS = {s: DATA_ROOT / s for s in ("train", "validation", "test")}
LOGGER.info("Real dataset root: %s", DATA_ROOT)
print("Found split folders:", {k: str(v) for k, v in SPLIT_DIRS.items()})

## Cell 4 — Dataset integrity checks

Confirm six class folders in all three splits; validate selected images (all images in FULL); record SHA-256 file hashes and fail on exact cross-split duplicates. `SMOKE` audits only selected files. Group/patient-level duplicate checks must be completed upstream to support an unbiased experiment.

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

records = []
issues = []
all_split_counts = {}
random_selector = random.Random(CFG.seed)
smoke_caps = {"train": CFG.smoke_train_per_class,
              "validation": CFG.smoke_val_per_class,
              "test": CFG.smoke_test_per_class}
for split, split_root in SPLIT_DIRS.items():
    expected = set(CLASSES)
    actual = {p.name for p in split_root.iterdir() if p.is_dir()}
    if actual != expected:
        raise RuntimeError(f"Wrong class directories in {split}: missing={expected-actual}, extra={actual-expected}")
    for label in CLASSES:
        image_paths = sorted(
            p for p in (split_root / label).rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        )
        if not image_paths:
            raise RuntimeError(f"No images found under {split_root / label}")
        all_split_counts[f"{split}/{label}"] = len(image_paths)
        if SMOKE:
            # Stratified fixed-seed sample, without changing any original split.
            k = min(smoke_caps[split], len(image_paths))
            if k <= 0:
                raise RuntimeError(f"SMOKE sampling found no images in {split}/{label}")
            image_paths = sorted(random_selector.sample(image_paths, k))
        for path in image_paths:
            rel = str(path.relative_to(DATA_ROOT))
            try:
                with Image.open(path) as im:
                    im.verify()
                with Image.open(path) as im:
                    im = ImageOps.exif_transpose(im).convert("RGB")
                    width, height = im.size
                    if width < 32 or height < 32:
                        issues.append({"path": rel, "issue": "very_small_image", "width": width, "height": height})
                    gray = np.asarray(im.resize((32, 32)).convert("L"), dtype=np.float32)
                    if float(gray.std()) < 3 or float(gray.mean()) < 8:
                        issues.append({"path": rel, "issue": "near_uniform_or_dark", "width": width, "height": height})
                digest = sha256_file(path) if CFG.verify_full_sha256 else "not_computed"
                records.append({
                    "split": split, "class_name": label, "class_id": CLASS_TO_IDX[label],
                    "relative_path": rel, "width": width, "height": height,
                    "file_size_bytes": path.stat().st_size, "sha256": digest
                })
            except (OSError, ValueError, UnidentifiedImageError) as exc:
                issues.append({"path": rel, "issue": "corrupt_or_unreadable", "error": str(exc)})
    LOGGER.info("Validated %s images: %d", split, sum(r["split"] == split for r in records))

manifest = pd.DataFrame.from_records(records)
manifest.to_csv(OUT / "reports" / "dataset_manifest.csv", index=False)
pd.DataFrame(issues, columns=["path", "issue", "width", "height", "error"]).to_csv(OUT / "reports" / "image_warnings.csv", index=False)

bad = [x for x in issues if x["issue"] == "corrupt_or_unreadable"]
if bad:
    raise RuntimeError(f"Found {len(bad)} corrupt images. See reports/image_warnings.csv. Fix the source dataset before training.")

if CFG.verify_full_sha256:
    overlap = (
        manifest.groupby("sha256", sort=False)
        .agg(n_splits=("split", "nunique"), count=("split", "size"))
        .reset_index()
    )
    leaked_hashes = set(overlap.loc[overlap.n_splits > 1, "sha256"])
    leakage = manifest[manifest.sha256.isin(leaked_hashes)].sort_values(["sha256", "split"])
    leakage.to_csv(OUT / "reports" / "cross_split_exact_duplicates.csv", index=False)
    if leaked_hashes:
        raise RuntimeError(f"Found {len(leaked_hashes)} exact cross-split duplicate hashes. See cross_split_exact_duplicates.csv")
    LOGGER.info("SHA-256 cross-split audit passed for %s files only.", "SAMPLED" if SMOKE else "ALL")
else:
    LOGGER.warning("Full SHA-256 disabled: exact cross-split duplicate check NOT performed.")

summary = (manifest.groupby(["split", "class_name"]).size()
           .unstack(fill_value=0).reindex(index=["train", "validation", "test"], columns=CLASSES))
summary.to_csv(OUT / "reports" / "split_class_counts.csv")
dataset_fingerprint = hashlib.sha256(
    manifest.sort_values("relative_path")[["relative_path", "sha256"]]
    .to_csv(index=False).encode("utf-8")
).hexdigest()
save_json(OUT / "reports" / "dataset_audit.json", {
    "root": str(DATA_ROOT), "fingerprint": dataset_fingerprint,
    "run_mode": RUN_MODE, "audit_scope": "sample_only" if SMOKE else "entire_dataset",
    "original_class_file_counts": all_split_counts,
    "total_images": int(len(manifest)), "split_sizes": {s: int(summary.loc[s].sum()) for s in summary.index},
    "class_names": CLASSES, "sha256_checked": CFG.verify_full_sha256,
    "exact_split_overlap_count": 0 if CFG.verify_full_sha256 else None,
    "image_warning_count": len(issues),
    "limitations": ("SMOKE verifies ONLY a selected subset; not a full dataset audit. " if SMOKE else "")
                   + "Exact hashes do not detect visually similar/augmented duplicates or patient overlap."
})
print(summary.to_string())
LOGGER.info("Dataset fingerprint: %s", dataset_fingerprint)
LOGGER.warning("Suspect/very dark images (review, not auto-removed): %d", len(issues))
if SMOKE:
    LOGGER.warning("SMOKE: audited %d sampled images, NOT the full dataset. Results are NOT for publication.", len(manifest))
else:
    LOGGER.info("FULL: audited %d images. Near-duplicate/patient checks need separate provenance.", len(manifest))


## Cell 5 — Train-only clinical-safe augmentation and image loaders

224×224 RGB images with aspect-ratio-preserving padding; horizontal flip p=0.5 and small ±7° rotation only for training. Deterministic validation/test; ImageNet channel mean/std. Avoid strong hue jitter, random erasing and aggressive crops that could alter disease evidence.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
PAD_COLOR = (128, 128, 128)

class AspectPad:
    def __init__(self, size: int, color=PAD_COLOR):
        self.size = size
        self.color = color
    def __call__(self, image):
        return ImageOps.pad(image, (self.size, self.size), method=Image.Resampling.BICUBIC, color=self.color, centering=(0.5, 0.5))

train_transform = transforms.Compose([
    AspectPad(CFG.image_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=7, interpolation=InterpolationMode.BILINEAR, fill=PAD_COLOR),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
valid_transform = transforms.Compose([
    AspectPad(CFG.image_size),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FixedImageDataset(Dataset):
    def __init__(self, items, root: Path, transform):
        self.items = items.reset_index(drop=True)
        self.root = root
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, index):
        row = self.items.iloc[index]
        image_path = self.root / row["relative_path"]
        with Image.open(image_path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            tensor = self.transform(image)
        return tensor, int(row["class_id"]), str(row["relative_path"])

train_ds = FixedImageDataset(manifest[manifest.split == "train"], DATA_ROOT, train_transform)
val_ds = FixedImageDataset(manifest[manifest.split == "validation"], DATA_ROOT, valid_transform)
test_ds = FixedImageDataset(manifest[manifest.split == "test"], DATA_ROOT, valid_transform)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

pin_memory = DEVICE.type == "cuda"

def make_loader(ds, *, shuffle=False):
    generator = torch.Generator().manual_seed(CFG.seed)
    return DataLoader(
        ds, batch_size=CFG.smoke_batch_size if SMOKE else CFG.batch_size, shuffle=shuffle, num_workers=CFG.num_workers,
        pin_memory=pin_memory, persistent_workers=(CFG.num_workers > 0),
        worker_init_fn=seed_worker, generator=generator
    )

train_loader = make_loader(train_ds, shuffle=True)
val_loader = make_loader(val_ds)
test_loader = make_loader(test_ds)
assert len(train_ds) and len(val_ds) and len(test_ds)
xs, ys, paths = next(iter(train_loader))
assert xs.ndim == 4 and xs.shape[1:] == (3, CFG.image_size, CFG.image_size)
assert ys.min() >= 0 and ys.max() < len(CLASSES)
LOGGER.info("Data loading sanity check passed: %s", tuple(xs.shape))

## Cell 6 — Official VMamba source, CUDA kernel and pretrained checkpoint

This is **not** a generic SSM approximation. The notebook clones the official source, records the exact Git SHA, and compiles the official `selective_scan_cuda_oflex` kernel if necessary. To keep kernel ABI correct, it uses Kaggle's installed PyTorch/CUDA rather than replacing PyTorch. Then it downloads and strictly loads `vmamba_tiny_s2l5`'s authentic 1000-class ImageNet checkpoint before constructing a new six-class classifier.

Official code: <https://github.com/MzeroMiko/VMamba>. Official Tiny checkpoint: `vssm_tiny_0230_ckpt_epoch_262.pth` in its GitHub release `%23v2cls`.

If kernel compilation, pretrained download, architecture import, or strict checkpoint loading fails, **stop and share the error**; do not go to FULL.

In [ ]:
# Official repository source is downloaded once. Preserve SHA + source code with each run.
OFFICIAL_REPO = OUTPUT_ROOT / "vmamba_official_source"
SOURCE_URL = "https://github.com/MzeroMiko/VMamba.git"
if not (OFFICIAL_REPO / ".git").is_dir():
    if OFFICIAL_REPO.exists():
        raise RuntimeError(f"Source directory exists but is not a Git repo: {OFFICIAL_REPO}")
    subprocess.check_call(["git", "clone", "--depth=1", SOURCE_URL, str(OFFICIAL_REPO)])
REPO_SHA = subprocess.check_output(["git", "-C", str(OFFICIAL_REPO), "rev-parse", "HEAD"], text=True).strip()
source_file = OFFICIAL_REPO / "vmamba.py"
csms_file = OFFICIAL_REPO / "classification" / "models" / "csms6s.py"
assert source_file.is_file() and csms_file.is_file(), "Missing official VMamba source files"
ARCH_DIR = OUT / "architecture"
ARCH_DIR.mkdir(exist_ok=True)
shutil.copy2(source_file, ARCH_DIR / "vmamba.py")
shutil.copy2(csms_file, ARCH_DIR / "csms6s.py")
(ARCH_DIR / "git_sha.txt").write_text(REPO_SHA + "\n", encoding="utf-8")
save_json(ARCH_DIR / "source_manifest.json", {
    "repository": SOURCE_URL, "git_sha": REPO_SHA,
    "model_builder": "vmamba_tiny_s2l5", "kernel": "selective_scan_cuda_oflex",
    "model_file_sha256": sha256_file(ARCH_DIR / "vmamba.py"),
    "scan_file_sha256": sha256_file(ARCH_DIR / "csms6s.py")
})
LOGGER.info("VMamba official source SHA: %s", REPO_SHA)

# Prefer a compatible already-importable CUDA extension. If absent, build the official one.
try:
    import selective_scan_cuda_oflex
    KERNEL_READY = True
    LOGGER.info("selective_scan_cuda_oflex already importable")
except ImportError:
    KERNEL_READY = False
if not KERNEL_READY:
    from torch.utils.cpp_extension import CUDA_HOME
    nvcc = Path(CUDA_HOME or "") / "bin" / "nvcc"
    if not CUDA_HOME or not nvcc.is_file():
        raise RuntimeError(
            "CUDA compilation toolkit (nvcc) was not found in Kaggle. "
            "The official VMamba CUDA selective-scan kernel cannot be built here. "
            "Please share this preflight output; do not run FULL or use a fake fallback."
        )
    env = os.environ.copy()
    env["MAX_JOBS"] = "2"      # Keep extension build RAM under control.
    major, minor = torch.cuda.get_device_capability(0)
    env["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
    kernel_path = OFFICIAL_REPO / "kernels" / "selective_scan"
    build_log = OUT / "kernel_build.log"
    command = [sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-deps", str(kernel_path)]
    LOGGER.info("Building official VMamba selective-scan CUDA kernel (log: %s)", build_log)
    with build_log.open("w", encoding="utf-8") as log_handle:
        result = subprocess.run(command, cwd=kernel_path, env=env, stdout=log_handle, stderr=subprocess.STDOUT, check=False)
    if result.returncode != 0:
        tail = build_log.read_text(encoding="utf-8", errors="replace")[-3500:]
        raise RuntimeError("VMamba CUDA kernel compilation failed. Open kernel_build.log; last lines:\n" + tail)
    importlib.invalidate_caches()
    import selective_scan_cuda_oflex

# Official single-file model implementation, patched to use *official* optimized oflex
# instead of its optional mamba_ssm kernel. Both are official VMamba code paths.
sys.path.insert(0, str(ARCH_DIR))
import vmamba as vmamba_official
import csms6s as vmamba_scan
if not vmamba_scan.WITH_SELECTIVESCAN_OFLEX:
    raise RuntimeError("Official oflex kernel could not load; no slow CPU/PyTorch substitute is allowed.")
vmamba_official.selective_scan_fn = vmamba_scan.selective_scan_fn

# Verify CUDA backend genuinely supports gradients before downloading a large checkpoint.
u = torch.randn(1, 8, 16, device=DEVICE, dtype=torch.float32, requires_grad=True)
delta = torch.randn_like(u, requires_grad=True)
A = -torch.ones(8, 1, device=DEVICE)
B = torch.randn(1, 4, 1, 16, device=DEVICE)
C = torch.randn(1, 4, 1, 16, device=DEVICE)
D = torch.ones(8, device=DEVICE)
scan_out = vmamba_scan.selective_scan_fn(u, delta, A, B, C, D, backend="oflex")
scan_out.sum().backward()
if not (torch.isfinite(scan_out).all() and torch.isfinite(u.grad).all()):
    raise RuntimeError("VMamba selective-scan kernel produced invalid output/gradient")
del u, delta, A, B, C, D, scan_out

# The official model method strictly loads a 1000-way ImageNet-1K state dict.
CKPT_URL = "https://github.com/MzeroMiko/VMamba/releases/download/%23v2cls/vssm_tiny_0230_ckpt_epoch_262.pth"
try:
    model = vmamba_official.vmamba_tiny_s2l5(pretrained=True)
except Exception as exc:
    raise RuntimeError(
        "The official VMamba-Tiny pretrained checkpoint did not load. "
        "Check Kaggle Internet and GitHub release access; do NOT use pretrained=False. "
        f"Original error: {exc}"
    ) from exc
if not (hasattr(model, "layers") and len(model.layers) == 4 and
        hasattr(model, "classifier") and isinstance(model.classifier.head, nn.Linear) and
        model.classifier.head.out_features == 1000):
    raise RuntimeError("Expected official VMamba-Tiny ImageNet-1K checkpoint/architecture")
model.classifier.head = nn.Linear(model.classifier.head.in_features, len(CLASSES))
model = model.to(DEVICE)
param_total = sum(p.numel() for p in model.parameters())
if not (15_000_000 <= param_total <= 40_000_000):
    raise RuntimeError(f"Unexpected VMamba-Tiny parameter count: {param_total}")
# Run full original model forward and backward using CUDA (no surrogate).
with torch.enable_grad():
    model.train()
    sanity_logits = model(xs[:1].to(DEVICE))
    if sanity_logits.shape != (1, len(CLASSES)) or not torch.isfinite(sanity_logits).all():
        raise RuntimeError("VMamba 6-class CUDA forward shape/nonfinite error")
    sanity_logits.float().square().mean().backward()
model.zero_grad(set_to_none=True)
print("Genuine VMamba-Tiny 6-class CUDA forward/backward PASSED; params:", param_total)
LOGGER.info("OFFICIAL VMamba Tiny loaded with ImageNet pretrained weights and working CUDA kernel")

## Cell 7 — Progressive VMamba fine-tuning groups

1. Train six-class classifier head only; 2. head + final VSSM stage `layers.3`; 3. head + last two stages `layers.2`, `layers.3`; 4. optional low-LR entire model. Stage-wise checkpoints prevent degradation. Class weights use only real training counts (not validation/test).

In [ ]:
train_counts = np.array([int((manifest.split.eq("train") & manifest.class_name.eq(c)).sum()) for c in CLASSES], dtype=np.float64)
if np.any(train_counts <= 0):
    raise RuntimeError("Every class must have training examples")
raw_weights = np.sqrt(train_counts.sum() / (len(CLASSES) * train_counts))
raw_weights = np.clip(raw_weights, 0.5, 2.0)
raw_weights /= raw_weights.mean()
class_weights = torch.tensor(raw_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights if CFG.use_class_weights else None)
LOGGER.info("Train counts: %s; loss weights: %s", train_counts.astype(int).tolist(), raw_weights.round(4).tolist())

STAGE_SPECS = [
    {"name": "head", "epochs": CFG.epochs_head, "lrs": {"head": 8e-4}},
    {"name": "stage4", "epochs": CFG.epochs_stage4,
     "lrs": {"last_stage": 1.5e-5, "head": 1e-4}},
    {"name": "stages3_4", "epochs": CFG.epochs_stages3_4,
     "lrs": {"penultimate_stage": 6e-6, "last_stage": 1e-5, "head": 6e-5}},
]
if CFG.full_fine_tune or (SMOKE and CFG.smoke_test_optional_full_stage):
    STAGE_SPECS.append({
        "name": "full", "epochs": CFG.epochs_full,
        "lrs": {"stem": 5e-7, "early_stage": 1e-6, "middle_stage": 3e-6,
                "penultimate_stage": 5e-6, "last_stage": 8e-6, "head": 3e-5}
    })
if SMOKE:
    STAGE_SPECS = [{**stage, "epochs": CFG.smoke_epochs_per_stage} for stage in STAGE_SPECS]
LOGGER.info("Effective stage schedule: %s", [(s["name"], s["epochs"]) for s in STAGE_SPECS])

def group_for_parameter(name: str) -> str:
    if name.startswith("classifier."):
        return "head"
    if name.startswith("patch_embed.") or name == "pos_embed":
        return "stem"
    if name.startswith("layers."):
        stage_number = int(name.split(".")[1])
        return {0: "early_stage", 1: "middle_stage", 2: "penultimate_stage", 3: "last_stage"}[stage_number]
    raise RuntimeError(f"Unrecognized VMamba parameter (architecture changed): {name}")

def prepare_stage(model, stage):
    buckets = defaultdict(list)
    for name, p in model.named_parameters():
        group = group_for_parameter(name)
        p.requires_grad_(group in stage["lrs"])
        if p.requires_grad:
            buckets[group].append(p)
    if set(buckets) != set(stage["lrs"]):
        raise RuntimeError(f"Missing optimizer groups: {set(stage['lrs']) - set(buckets)}")
    optimizer = torch.optim.AdamW(
        [{"params": buckets[group], "lr": lr, "group_name": group}
         for group, lr in stage["lrs"].items()], weight_decay=CFG.weight_decay
    )
    count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    LOGGER.info("Stage %s trainable params=%d, LRs=%s", stage["name"], count, stage["lrs"])
    return optimizer

def freeze_bn_running_stats(module):
    if isinstance(module, nn.modules.batchnorm._BatchNorm):
        module.eval()

def metric_dict(y_true, y_pred, loss):
    return {
        "loss": float(loss),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=list(range(len(CLASSES))), average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

model.eval()
with torch.inference_mode():
    logits = model(xs[:1].to(DEVICE))
assert logits.shape == (1, len(CLASSES))
del xs, ys, paths, logits
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## Cell 8 — CUDA AMP, safe checkpoints and evaluation

Use AMP on the T4 if compatible with the official CUDA scan kernel; in case of dtype compatibility problems turn `use_amp=False` in the configuration and rerun SMOKE. Save the complete official VMamba state dict, plus source commit, preprocessing and mapping. Checkpoints use only safe tensor/primitive metadata for `torch.load(weights_only=True)`.

In [ ]:
AMP_ENABLED = CFG.use_amp and DEVICE.type == "cuda"
SCALER = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
HISTORY_PATH = OUT / "reports" / "history.csv"
history_rows = []

def amp_context():
    return torch.amp.autocast(device_type="cuda", enabled=True) if AMP_ENABLED else nullcontext()

def run_epoch(model, loader, *, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    if is_train:
        model.apply(freeze_bn_running_stats)
    running_loss, seen = 0.0, 0
    truths, guesses, prediction_rows = [], [], []
    context = torch.enable_grad() if is_train else torch.inference_mode()
    with context:
        for batch_x, batch_y, batch_paths in loader:
            batch_x = batch_x.to(DEVICE, non_blocking=pin_memory)
            batch_y = batch_y.to(DEVICE, non_blocking=pin_memory)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
            with amp_context():
                batch_logits = model(batch_x)
                loss = criterion(batch_logits, batch_y)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite loss detected; stopped to protect checkpoints")
            if is_train:
                SCALER.scale(loss).backward()
                SCALER.unscale_(optimizer)
                nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), CFG.grad_clip_norm)
                SCALER.step(optimizer)
                SCALER.update()
            probs = torch.softmax(batch_logits.detach().float(), dim=1)
            predictions = probs.argmax(dim=1)
            y = batch_y.detach().cpu().tolist()
            yhat = predictions.detach().cpu().tolist()
            truths.extend(y)
            guesses.extend(yhat)
            if not is_train:
                all_probs = probs.detach().cpu().numpy()
                for row_path, y_true, y_pred, probability_vector in zip(batch_paths, y, yhat, all_probs):
                    row = {"relative_path": row_path, "true_id": int(y_true), "pred_id": int(y_pred),
                           "true_label": CLASSES[y_true], "pred_label": CLASSES[y_pred],
                           "confidence": float(probability_vector[y_pred])}
                    row.update({f"prob_{c}": float(probability_vector[i]) for i, c in enumerate(CLASSES)})
                    prediction_rows.append(row)
            batch_size = len(batch_y)
            running_loss += float(loss.detach().item()) * batch_size
            seen += batch_size
    assert seen > 0
    metrics = metric_dict(truths, guesses, running_loss / seen)
    return metrics, prediction_rows

def save_checkpoint(file_path, model, stage, epoch, val_metrics):
    payload = {
        "model_name": CFG.model_name, "run_mode": RUN_MODE,
        "checkpoint_purpose": "pipeline_verification_only" if SMOKE else "research_training",
        "weights_origin": CFG.imagenet_weights,
        "official_model_builder": "vmamba_tiny_s2l5",
        "official_repo_sha": REPO_SHA,
        "pretrained_checkpoint_url": CKPT_URL,
        "architecture": "official_vmamba.VSSM.s2l5",
        "model_state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        "num_classes": len(CLASSES), "classes": CLASSES, "class_to_idx": CLASS_TO_IDX,
        "preprocessing": {"input_size": CFG.image_size, "resize": "aspect_ratio_pad",
                          "pad_rgb": list(PAD_COLOR), "mean": list(IMAGENET_MEAN),
                          "std": list(IMAGENET_STD), "exif_transpose": True},
        "stage": stage, "stage_epoch": int(epoch), "validation": val_metrics,
        "dataset_fingerprint": dataset_fingerprint,
        "random_seed": CFG.seed,
        "torch_version": str(torch.__version__),
        "created_utc": datetime.now(timezone.utc).isoformat()
    }
    temp = Path(str(file_path) + ".tmp")
    torch.save(payload, temp)
    os.replace(temp, file_path)

# Compare validation macro-F1 first; on exact ties, prefer lower validation loss.
def better(candidate, incumbent, eps=1e-8):
    if incumbent is None:
        return True
    if candidate["macro_f1"] > incumbent["macro_f1"] + eps:
        return True
    return (abs(candidate["macro_f1"] - incumbent["macro_f1"]) <= eps
            and candidate["loss"] < incumbent["loss"] - eps)

## Cell 9 — Training orchestration and early stopping

For head → last two blocks → last four blocks (optional full stage), train from previous stage's **best validation checkpoint**, not its final epoch. Track per-epoch metrics, current LRs, elapsed seconds, save best-in-stage and best-global checkpoints, and select the global best solely from validation macro-F1. Any exception is recorded in `error_traceback.txt`.

In [ ]:
def safe_load_checkpoint(file_path, map_location="cpu"):
    try:
        return torch.load(file_path, map_location=map_location, weights_only=True)
    except TypeError as exc:
        raise RuntimeError("This notebook requires PyTorch with torch.load(weights_only=True).") from exc

def run_all_stages():
    global_best = None
    global_path = OUT / "checkpoints" / "best_model.pt"
    stage_results = []
    for stage in STAGE_SPECS:
        name = stage["name"]
        LOGGER.info("=== BEGIN STAGE: %s | max epochs: %d ===", name, stage["epochs"])
        optimizer = prepare_stage(model, stage)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, stage["epochs"]), eta_min=0)
        stage_best = None
        stage_path = OUT / "checkpoints" / f"best_{name}.pt"
        stagnant = 0
        for epoch in range(1, stage["epochs"] + 1):
            tick = time.perf_counter()
            train_metrics, _ = run_epoch(model, train_loader, optimizer=optimizer)
            validation_metrics, _ = run_epoch(model, val_loader)
            row = {"stage": name, "epoch_in_stage": epoch,
                   "train_loss": train_metrics["loss"], "train_accuracy": train_metrics["accuracy"],
                   "train_macro_f1": train_metrics["macro_f1"],
                   "val_loss": validation_metrics["loss"], "val_accuracy": validation_metrics["accuracy"],
                   "val_macro_f1": validation_metrics["macro_f1"],
                   "val_balanced_accuracy": validation_metrics["balanced_accuracy"],
                   "seconds": round(time.perf_counter() - tick, 2)}
            row.update({f"lr_{group['group_name']}": float(group["lr"]) for group in optimizer.param_groups})
            history_rows.append(row)
            pd.DataFrame(history_rows).to_csv(HISTORY_PATH, index=False)
            LOGGER.info("%s epoch %02d/%02d | train_loss=%.4f val_loss=%.4f val_macro_f1=%.4f val_acc=%.4f sec=%.1f",
                        name, epoch, stage["epochs"], train_metrics["loss"], validation_metrics["loss"],
                        validation_metrics["macro_f1"], validation_metrics["accuracy"], row["seconds"])
            if better(validation_metrics, stage_best):
                stage_best = dict(validation_metrics)
                save_checkpoint(stage_path, model, name, epoch, validation_metrics)
                stagnant = 0
            else:
                stagnant += 1
            if better(validation_metrics, global_best):
                global_best = dict(validation_metrics)
                save_checkpoint(global_path, model, name, epoch, validation_metrics)
                LOGGER.info("Updated global best at %s epoch %d: val_macro_f1=%.4f", name, epoch, validation_metrics["macro_f1"])
            scheduler.step()
            if epoch >= CFG.min_epochs_per_stage and stagnant >= CFG.patience:
                LOGGER.info("Early stopping %s after epoch %d", name, epoch)
                break
        if stage_best is None:
            raise RuntimeError(f"Stage {name} completed without a checkpoint")
        stage_results.append({"stage": name, "best_val": stage_best, "checkpoint": str(stage_path), "completed_epochs": epoch})
        save_json(OUT / "reports" / "stage_results.json", stage_results)
        model.load_state_dict(safe_load_checkpoint(stage_path, DEVICE)["model_state_dict"], strict=True)
        LOGGER.info("Reloaded best checkpoint from stage %s before next stage.", name)
    if not global_path.is_file():
        raise RuntimeError("No globally selected model checkpoint found")
    return global_path, stage_results

save_json(OUT / "run_status.json", {"status": "smoke_running" if SMOKE else "full_running", "run_mode": RUN_MODE, "started_utc": datetime.now(timezone.utc).isoformat()})
try:
    BEST_PATH, STAGE_RESULTS = run_all_stages()
    save_json(OUT / "run_status.json", {"status": "smoke_trained_not_yet_evaluated" if SMOKE else "full_trained_not_yet_evaluated", "run_mode": RUN_MODE, "best_checkpoint": str(BEST_PATH),
                                        "updated_utc": datetime.now(timezone.utc).isoformat()})
except Exception as exc:
    tb = traceback.format_exc()
    (OUT / "error_traceback.txt").write_text(tb, encoding="utf-8")
    save_json(OUT / "run_status.json", {"status": "failed_during_training", "run_mode": RUN_MODE, "error": str(exc)})
    LOGGER.exception("Training failed")
    raise

## Cell 10 — One-time held-out test evaluation

Reload the globally selected checkpoint, score validation again, then score the untouched real test split once. Save per-image probabilities, confusion matrix, classification report, complete metrics JSON, and label mapping. Smoke scores are debugging diagnostics, never paper results.

In [ ]:
best_payload = safe_load_checkpoint(BEST_PATH, DEVICE)
assert best_payload["class_to_idx"] == CLASS_TO_IDX
assert best_payload["dataset_fingerprint"] == dataset_fingerprint
model.load_state_dict(best_payload["model_state_dict"], strict=True)
model.to(DEVICE).eval()

selected_val_metrics, selected_val_rows = run_epoch(model, val_loader)
val_predictions = pd.DataFrame(selected_val_rows)
val_predictions.to_csv(OUT / "reports" / "validation_predictions.csv", index=False)

if SMOKE:
    LOGGER.warning("SMOKE TEST metrics are pipeline diagnostics, NOT held-out research results!")
else:
    LOGGER.info("FULL: held-out test evaluation begins; do not change training settings from test feedback.")
test_metrics, test_rows = run_epoch(model, test_loader)
test_predictions = pd.DataFrame(test_rows)
test_predictions.to_csv(OUT / "reports" / "test_predictions.csv", index=False)

truths = test_predictions.true_id.astype(int).tolist()
preds = test_predictions.pred_id.astype(int).tolist()
report_dict = classification_report(truths, preds, labels=list(range(len(CLASSES))),
                                    target_names=CLASSES, zero_division=0, output_dict=True)
report_text = classification_report(truths, preds, labels=list(range(len(CLASSES))),
                                    target_names=CLASSES, zero_division=0)
save_json(OUT / "reports" / "test_classification_report.json", report_dict)
(OUT / "reports" / "test_classification_report.txt").write_text(report_text, encoding="utf-8")
cm = confusion_matrix(truths, preds, labels=list(range(len(CLASSES))))
pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(OUT / "reports" / "test_confusion_matrix.csv")

metrics_output = {
    "model": CFG.model_name, "training_data": "real_only",
    "run_mode": RUN_MODE, "evaluation_scope": "smoke_subset_debug_only" if SMOKE else "full_holdout_test",
    "publishable": not SMOKE,
    "selection_metric": "validation_macro_f1",
    "selected_stage": best_payload["stage"], "selected_epoch_within_stage": best_payload["stage_epoch"],
    "best_validation_at_selection": best_payload["validation"],
    "validation_after_reload": selected_val_metrics,
    "test": test_metrics,
    "test_classification_report": report_dict,
    "dataset_fingerprint": dataset_fingerprint,
    "test_samples": len(truths), "classes": CLASSES,
    "warning": ("SMOKE SUBSET METRICS ARE INVALID FOR SCIENTIFIC REPORTING. " if SMOKE else "")
               + "Research-only image classification. Test set must not be used for model selection."
}
save_json(OUT / "reports" / "final_metrics.json", metrics_output)
LOGGER.info("SELECTED: %s epoch %d", best_payload["stage"], best_payload["stage_epoch"])
LOGGER.info("FINAL TEST: %s", test_metrics)
print("\n=== " + ("SMOKE / NON-REPORTABLE" if SMOKE else "FULL HOLDOUT") + " TEST REPORT ===\n", report_text)

## Cell 11 — Visual plots

Save separate validation/training curves and confusion matrices for comparison with the other model notebooks. Graphs use the same six-class ordering and file naming as the ResNet-50/ConvNeXt V2 notebooks.

In [ ]:
hist = pd.read_csv(HISTORY_PATH)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for stage_name, rows in hist.groupby("stage", sort=False):
    step = rows.index.to_numpy() + 1
    axes[0].plot(step, rows.train_loss, marker=".", label=f"{stage_name} train")
    axes[0].plot(step, rows.val_loss, marker=".", linestyle="--", label=f"{stage_name} val")
    axes[1].plot(step, rows.val_macro_f1, marker="o", label=stage_name)
axes[0].set(title="Training and validation loss", xlabel="Global epoch", ylabel="Cross-entropy loss")
axes[1].set(title="Validation macro-F1", xlabel="Global epoch", ylabel="Macro-F1", ylim=(0, 1.02))
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "plots" / "learning_curves.png", dpi=160)
plt.close(fig)

for normalized in (False, True):
    plot_cm = cm.astype(float)
    if normalized:
        denom = plot_cm.sum(axis=1, keepdims=True)
        plot_cm = np.divide(plot_cm, denom, out=np.zeros_like(plot_cm), where=denom > 0)
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(plot_cm, cmap="Blues", vmin=0, vmax=1 if normalized else None)
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=35, ha="right")
    ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title("Held-out test confusion matrix" + (" (row-normalized)" if normalized else " (counts)"))
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            text_value = f"{plot_cm[i, j]:.2f}" if normalized else str(int(plot_cm[i, j]))
            ax.text(j, i, text_value, ha="center", va="center", fontsize=9,
                    color="white" if plot_cm[i, j] > (0.5 if normalized else cm.max()/2) else "black")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(OUT / "plots" / ("confusion_normalized.png" if normalized else "confusion_counts.png"), dpi=160)
    plt.close(fig)
LOGGER.info("Saved learning curves and confusion matrices.")

## Cell 12 — Independent VMamba checkpoint reload and prediction

Package original VMamba source, restore the saved state dict into the **same** `vmamba_tiny_s2l5` architecture, and confirm model predictions agree with validation output. VMamba requires its optimized selective-scan kernel in every new inference environment; the ZIP provides source/metadata but the compiled CUDA `.so` is **not portable** between Python/CUDA versions.

In [ ]:
PREPROCESS_META = best_payload["preprocessing"]
save_json(OUT / "preprocessing.json", PREPROCESS_META)
inference_script = r'''"""Reusable DermaLens VMamba-Tiny inference (GPU, research only).

Usage: python inference_example.py --checkpoint /path/checkpoints/best_model.pt --image photo.jpg

Requires torch/timm/fvcore and a matching VMamba official selective_scan_cuda_oflex
extension. Copy the entire artifact folder, including architecture/*.py, and
follow README.md to reinstall the extension in a new Kaggle environment.
"""
import argparse, json, sys
from pathlib import Path
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image, ImageOps


def load_classifier(checkpoint_path, device="cuda"):
    ckpt = Path(checkpoint_path)
    artifact_root = ckpt.resolve().parent.parent
    src = artifact_root / "architecture"
    if not (src / "vmamba.py").is_file():
        raise FileNotFoundError(f"Missing official VMamba source in {src}")
    if not torch.cuda.is_available():
        raise RuntimeError("VMamba requires a CUDA GPU and compatible selective-scan extension")
    sys.path.insert(0, str(src))
    try:
        import selective_scan_cuda_oflex
        import vmamba as official
        import csms6s as scan
    except Exception as exc:
        raise RuntimeError("Install/build matching official VMamba selective-scan CUDA extension first") from exc
    if not scan.WITH_SELECTIVESCAN_OFLEX:
        raise RuntimeError("selective_scan_cuda_oflex is unavailable; no fallback")
    official.selective_scan_fn = scan.selective_scan_fn
    payload = torch.load(str(ckpt), map_location="cpu", weights_only=True)
    if payload["official_model_builder"] != "vmamba_tiny_s2l5":
        raise RuntimeError("Unexpected checkpoint architecture")
    model = official.vmamba_tiny_s2l5(pretrained=False)
    model.classifier.head = nn.Linear(model.classifier.head.in_features, len(payload["classes"]))
    model.load_state_dict(payload["model_state_dict"], strict=True)
    model.to(device).eval()
    return model, payload


def make_preprocess(meta):
    size = int(meta["input_size"])
    color = tuple(meta["pad_rgb"])
    def pad(image):
        return ImageOps.pad(image, (size, size), method=Image.Resampling.BICUBIC, color=color)
    return transforms.Compose([pad, transforms.ToTensor(), transforms.Normalize(meta["mean"], meta["std"])])

@torch.inference_mode()
def predict(checkpoint_path, image_path, device="cuda"):
    model, payload = load_classifier(checkpoint_path, device)
    preprocessing = make_preprocess(payload["preprocessing"])
    with Image.open(image_path) as im:
        tensor = preprocessing(ImageOps.exif_transpose(im).convert("RGB")).unsqueeze(0).to(device)
    probabilities = torch.softmax(model(tensor).float(), dim=1)[0].cpu().tolist()
    idx = max(range(len(probabilities)), key=probabilities.__getitem__)
    return {"prediction": payload["classes"][idx],
            "probabilities": {name: float(p) for name, p in zip(payload["classes"], probabilities)},
            "caution": "Research only — not for clinical diagnosis"}

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--image", required=True)
    args = parser.parse_args()
    print(json.dumps(predict(args.checkpoint, args.image), indent=2))
'''
(OUT / "inference_example.py").write_text(inference_script, encoding="utf-8")
compile(inference_script, str(OUT / "inference_example.py"), "exec")
reuse_metadata = {
    "artifact_schema_version": "dermalens-classifier-v1",
    "model": CFG.model_name, "run_mode": RUN_MODE,
    "publishable": not SMOKE,
    "checkpoint_relative_path": "checkpoints/best_model.pt",
    "class_map_relative_path": "class_to_idx.json",
    "preprocess_relative_path": "preprocessing.json",
    "metrics_relative_path": "reports/final_metrics.json",
    "test_predictions_relative_path": "reports/test_predictions.csv",
    "architecture_source": "architecture/vmamba.py",
    "kernel_source_repository": SOURCE_URL,
    "official_repo_sha": REPO_SHA,
    "kernel_required": "selective_scan_cuda_oflex",
    "dataset_fingerprint": dataset_fingerprint,
    "test_set_locked": True,
    "note": "Attach artifact output and compile matching kernel in new GPU environment. No pretrained download on reload."
}
save_json(OUT / "artifact_manifest.json", reuse_metadata)
independent_ns = {"__name__": "dermalens_reuse_check"}
exec(compile(inference_script, "inference_example.py", "exec"), independent_ns)
probe_image = DATA_ROOT / str(val_predictions.iloc[0]["relative_path"])
reuse_out = independent_ns["predict"](str(BEST_PATH), str(probe_image), device="cuda")
expected = val_predictions.iloc[0]
for class_name in CLASSES:
    if abs(float(reuse_out["probabilities"][class_name]) - float(expected[f"prob_{class_name}"])) > 5e-3:
        raise AssertionError(f"Independent GPU inference mismatch for {class_name}")
save_json(OUT / "reports" / "inference_reload_check.json", {
    "passed": True, "mode": RUN_MODE, "image": str(expected["relative_path"]),
    "predicted_class": reuse_out["prediction"], "probability_tolerance": 0.005,
    "note": "Independent GPU model reload and preprocessing agree with validation predictions."
})
LOGGER.info("Independent official VMamba inference script and checkpoint reload verified.")

## Cell 13 — Save logs, environment, metrics and reusable ZIP

The ZIP bundles official model source (`architecture/`), checkpoint, configuration, per-image predictions, figures, train history, CUDA build log if generated, version information, and inference script. **It does not include a portable compiled GPU extension**. In another Kaggle notebook, attach the ZIP/output and rebuild the extension against that environment before loading the model.

In [ ]:
package_info = {
    "python": sys.version.split()[0],
    "torch": str(torch.__version__),
    "torchvision": __import__("torchvision").__version__,
    "timm": timm.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pillow": Image.__version__ if hasattr(Image, "__version__") else "unknown",
    "sklearn": __import__("sklearn").__version__,
    "cuda_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "official_vmamba_repo_sha": REPO_SHA,
    "vmamba_pretrained_url": CKPT_URL,
    "optimized_cuda_kernel": "selective_scan_cuda_oflex",
    "stage_count": len(STAGE_SPECS), "run_mode": RUN_MODE,
    "audit_scope": "subset_only" if SMOKE else "full_dataset",
    "best_stage": best_payload["stage"],
    "best_stage_epoch": best_payload["stage_epoch"],
    "dataset_fingerprint": dataset_fingerprint,
}
save_json(OUT / "environment.json", package_info)

readme = f"""# DermaLens VMamba-Tiny S2L5 Real-Only Progressive Fine-Tuning

## MODE: {RUN_MODE}
- **RESEARCH STATUS:** {"SMOKE CHECK ONLY — NOT REPORTABLE" if SMOKE else "FULL REAL-DATA TRAINING"}
- To run the complete experiment, change `RUN_MODE = "SMOKE"` to `RUN_MODE = "FULL"` in the central settings cell and Save & Run All.
- SMOKE audits only sampled files and gives non-publishable debugging metrics.

## Checkpoint
- `checkpoints/best_model.pt`: selected by validation macro-F1 (ties: lower validation loss).
- `checkpoints/best_<stage>.pt`: best checkpoint from each completed stage.
- Selected stage: **{best_payload['stage']}**, epoch **{best_payload['stage_epoch']}**.
- Dataset fingerprint: `{dataset_fingerprint}`.

## Inputs and outputs
- Input: frozen `train/`, `validation/`, `test/` Kaggle real dataset.
- `config.json`: reproducible training configuration.
- `preprocessing.json`, `class_to_idx.json`: loader contract.
- `artifact_manifest.json`: downstream evaluation adapter contract.
- `reports/dataset_manifest.csv`, `reports/split_class_counts.csv`: immutable input audit.
- `reports/history.csv`, `reports/stage_results.json`: complete training history.
- `reports/final_metrics.json`, `reports/test_classification_report.*`: final evaluations.
- `reports/test_predictions.csv`: all per-image test probabilities.
- `plots/*.png`: training curves and confusion matrices.
- `training.log`: complete application log.
- `inference_example.py`: independent GPU inference (no pretrained download).
- `architecture/vmamba.py` and `architecture/csms6s.py`: exact official model source.
- `architecture/git_sha.txt`: pinned upstream commit of this run.
- `kernel_build.log` (if present): compilation log. The optimized CUDA extension must be installed again in new environments.

## Loading the artifact in another Kaggle notebook
1. Add this **notebook's saved output** as Input.
2. Locate `checkpoints/best_model.pt` under `/kaggle/input/`.
3. Compile `kernels/selective_scan` from the exact official repository commit listed in `architecture/git_sha.txt` for the current Python/CUDA/PyTorch runtime.
4. Use `inference_example.py` and the bundled `architecture/*.py` (and `torch.load(path, map_location='cpu', weights_only=True)`).

## Research precautions
- This is research software, **not a validated clinical diagnostic system**.
- The held-out test was evaluated only once for the finalized run; do not tune to the test results.
- Exact SHA-256 cross-split matching cannot detect near-duplicates or same-patient overlap.
- Default run uses one seed; additional independent seeds are needed for uncertainty estimates.
- GPU custom kernels may not be bitwise deterministic.
- This notebook will fail early if the official optimized CUDA kernel cannot compile/import; do not replace VMamba with a different model.
- Results cannot substantiate patient-level generalization without patient-disjoint identifiers.
"""
(OUT / "README.md").write_text(readme, encoding="utf-8")
zip_path = OUTPUT_ROOT / f"{RUN_OUTPUT_NAME}_artifact.zip"
# Finalize metadata, then archive outputs in one pass.
save_json(OUT / "run_status.json", {
    "status": "smoke_passed" if SMOKE else "full_completed", "run_mode": RUN_MODE,
    "publishable": not SMOKE, "updated_utc": datetime.now(timezone.utc).isoformat(),
    "checkpoint": str(BEST_PATH), "zip": str(zip_path),
    "selected_stage": best_payload["stage"], "val_macro_f1": float(best_payload["validation"]["macro_f1"]),
    "test_macro_f1": float(test_metrics["macro_f1"])
})
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED, compresslevel=3,
                     allowZip64=True) as bundle:
    for path in sorted(OUT.rglob("*")):
        if path.is_file() and not path.name.endswith(".tmp"):
            bundle.write(path, arcname=str(path.relative_to(OUT)))

print("\n=== " + RUN_MODE + " RUN COMPLETED ===")
print("Selected stage:", best_payload["stage"])
print("Selected validation macro-F1:", round(best_payload["validation"]["macro_f1"], 4))
print("Final test macro-F1:", round(test_metrics["macro_f1"], 4))
print("Best checkpoint:", BEST_PATH)
print("Reusable artifact ZIP:", zip_path, "size MiB:", round(zip_path.stat().st_size / 2**20, 1))
LOGGER.info("%s pipeline completed; outputs exported.", RUN_MODE)
if SMOKE:
    print("SMOKE PASSED: code paths executed on a tiny subset. NOT a full dataset audit or research result.")
    print('NEXT: Change RUN_MODE = "SMOKE" to RUN_MODE = "FULL" in Cell 2, then Save & Run All.')

## After the Kaggle run

1. **Upload this notebook** to another Kaggle account.
2. Attach **Facial Dermatology Image Dataset**, not the old unsplit `facial-skin-conditions-real-image-collection` dataset. The correct Kaggle owner/slug is `shanmukhadattaboda/facial-dermatology-image-dataset`.
3. Select **GPU T4** and enable **Internet**. The first run may compile an official CUDA extension and download a pretrained checkpoint.
4. Leave `RUN_MODE = "SMOKE"`. Use **Save Version → Save & Run All**.
5. Only if the final cell says `SMOKE PASSED`, change **one line** to `RUN_MODE = "FULL"`, then Save & Run All again. Do **not** train FULL if the CUDA build/import or checkpoint download fails.
6. Save the `vmamba_tiny_s2l5_real_progressive_v1_FULL_artifact.zip` under Kaggle Output. It contains checkpoints, code, preprocessing metadata, logs and predictions.

**Expected FULL outputs:** `checkpoints/best_model.pt`, `best_head.pt`, `best_stage4.pt`, `best_stages3_4.pt`, `reports/history.csv`, `reports/final_metrics.json`, `reports/test_predictions.csv`, `architecture/`, `inference_example.py`, `training.log`, `run_status.json`, and the artifact ZIP. Optional `best_full.pt` is generated only if optional full fine-tuning is enabled.

**Important:** VMamba uses a custom CUDA extension. If Kaggle cannot compile it with the available CUDA/PyTorch combination, please share `kernel_build.log`. Do not treat a substituted architecture or CPU fallback as a genuine VMamba smoke pass. Cross-split SHA-256 audit is exact-only; patient-level overlap and near-duplicates must be handled upstream. SMOKE performance metrics are not scientifically valid.
